# Department of Labor appreticeship program data

This short notebook analyzes data from the DoL "Apprenticeship Participant Characteristics and Training Outcomes" data, available here: https://data.dol.gov/datasets/10264 

In [ ]:
# Import libraries
import requests, pandas as pd, json, warnings

warnings.filterwarnings("ignore")

with open("api_key.txt", "r") as file:
    API_KEY = file.read().strip()

st = pd.read_csv("state_abbr.csv", sep="\t")

Metadata:

In [ ]:
# def create_filter_condition(field, operator, value):
#     return {"and": [{"field": field, "operator": operator, "value": value}]}
def create_filter_condition(field, operator, value):
    condition = {"field": field, "operator": operator, "value": value}

    return json.dumps(condition)


params = {"X-API-KEY": API_KEY}

# Get metadata
metadata_url = "https://apiprod.dol.gov/v4/get/ETA/apprenticeship_data/json/metadata"
metadata_request = requests.get(metadata_url, params=params, verify=False)
metadata_json = metadata_request.json()
metadata = pd.DataFrame(metadata_json)
metadata

In [ ]:
# Build filters as dicts, not strings
filter_1 = create_filter_condition("union_y_n", "eq", "1")
filter_2 = create_filter_condition("value", "gt", "999")
filter_3 = create_filter_condition("industry", "in", ["A", "B", "C"])
filter_4 = create_filter_condition("industry", "like", "%A%")

url = "https://apiprod.dol.gov/v4/get/ETA/apprenticeship_data/json"

headers = {"X-API-KEY": API_KEY}

params = {
    "X-API-KEY": API_KEY,
    "limit": "1000",
    "offset": "0",
    # Pass the filter as a JSON string in the query param
    "filter_object": filter_1,  # Swap filter_1 with any of the other filters (2-4) to see different results.
}

request = requests.get(url, headers=headers, params=params, verify=False)
print(f"Request is {request}")
data_json = request.json()["data"]
data = pd.DataFrame(data_json)

Preview of the data:

In [ ]:
data.head()

Now we'll loop through each state and grab each state's data.

In [ ]:
st_list = sorted(st.abbrv.tolist())

In [ ]:
n = len(st_list)
i = 1

for s in st_list:
    print(f"On state {i} of {n}, {s}")
    filt = create_filter_condition("state", "eq", s)

    params = {
        "X-API-KEY": API_KEY,
        "limit": "1000",
        "offset": "0",
        # Pass the filter as a JSON string in the query param
        "filter_object": filt,  # Swap filter_1 with any of the other filters (2-4) to see different results.
    }

    request = requests.get(url, headers=headers, params=params, verify=False)
    print(f"Request is {request}")
    data_json = request.json()["data"]
    data = pd.DataFrame(data_json)

    if s == "AK":
        df = data
    else:
        df = pd.concat([df, data], ignore_index=True)

    i += 1

In [ ]:
df